# 04 Pure factor portfolios

Each endpoint is fetched once. The weekly and historical handlers accept no limit parameter. Tables summarize books rather than expanding every holding.

These are dated research records, not investment advice. Each panel refreshes on a schedule; each panel shows its source date.

Free access grants no redistribution rights. **Do not republish the data you fetch.**

These are member research routes (beta); they are not yet listed in `/api/v1/catalog`.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'notebooks' / '_shared.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'notebooks'))
sys.path.insert(0, str(ROOT / 'python' / 'src'))
from _shared import fetch_panel, get_api_key, make_client, nested
client, catalogue = make_client()
api_key = get_api_key()
book_columns = {
    'factor': 'The style associated with this research book.',
    'status': 'The publication state for this book.',
    'source_state': 'The state supplied by the underlying publication.',
    'n_holdings': 'The number of holdings in the full response.',
    'formation_date': 'The formation date for the book.',
    'week_end_session': 'The final session in the stated week.',
    'book_family': 'The declared research-book family.',
}
def archive_rows(payload, container):
    data = payload.get('data') or {}
    archive = data.get(container) or {}
    return [{'factor': book.get('factor'), 'status': book.get('status'), 'source_state': book.get('source_state'), 'n_holdings': book.get('n_holdings'), 'formation_date': archive.get('formation_date'), 'week_end_session': archive.get('week_end_session'), 'book_family': archive.get('book_family')} for book in archive.get('books') or []]

In [ ]:
weekly_frame = fetch_panel('Current weekly research books', lambda: client.member_weekly_pure_books(api_key), lambda p: archive_rows(p, 'portfolio'), book_columns, source_date=lambda p: nested(p, 'data', 'portfolio', 'formation_date'), status=lambda p: nested(p, 'data', 'status'), message=lambda p: nested(p, 'data', 'message'))

In [ ]:
historical_frame = fetch_panel('Historical research books', lambda: client.member_historical_pure_books(api_key), lambda p: archive_rows(p, 'archive'), book_columns, source_date=lambda p: nested(p, 'data', 'archive', 'formation_date'), status=lambda p: nested(p, 'data', 'status'), message=lambda p: nested(p, 'data', 'message'))

In [ ]:
three_columns = {
    'factor': 'The style associated with this research book.',
    'status': 'The publication state for this book.',
    'n_holdings': 'The number of holdings returned for this book.',
    'sum_weight': 'The sum of returned holding weights.',
    'holdings_truncated': 'Whether the holdings limit shortened the response.',
    'formation_date': 'The selected formation date.',
    'entry_session': 'The entry session attached to the selected formation.',
    'week_state': 'Whether the selected week is open or complete.',
    'return_state': 'The maturity state attached to the weekly record.',
}
def three_rows(payload):
    data = payload.get('data') or {}
    return [{'factor': book.get('factor'), 'status': book.get('status'), 'n_holdings': book.get('n_holdings'), 'sum_weight': book.get('sum_weight'), 'holdings_truncated': book.get('holdings_truncated'), 'formation_date': data.get('formation_date'), 'entry_session': data.get('entry_session'), 'week_state': data.get('week_state'), 'return_state': data.get('return_state')} for book in data.get('portfolios') or []]
three_frame = fetch_panel('Three-model portfolio books', lambda: client.member_three_model_portfolios(api_key, holdings=25), three_rows, three_columns, source_date=lambda p: nested(p, 'data', 'formation_date'), status=lambda p: nested(p, 'data', 'status'), message=lambda p: nested(p, 'data', 'message'))